# Can ANY model beat naive here? - gold

The transformer only beats naive (predict-the-average) by 0.8%. This tests whether that's a fundamental ceiling on what's learnable from this data, or a transformer-specific training problem - by trying the simplest possible models on the exact same data and target, with no attention, no neural network at all.

Two simple models:
1. **Linear regression on all 30 raw window values** - same input as the transformer, simplest possible model class
2. **Linear regression on one feature**: the window's average absolute return (a simple "recent volatility predicts next volatility" test - the same basic idea GARCH exploits via its beta term)

In [1]:
import numpy as np
import pandas as pd

WINDOW_SIZE = 30

gold = pd.read_csv("../data/gold_futures.csv", skiprows=[1, 2], index_col=0, parse_dates=True)
returns = gold["Close"].pct_change().dropna() * 100


def create_windows(returns, window_size):
    values = returns.values
    X, y = [], []
    for start in range(len(values) - window_size):
        end = start + window_size
        X.append(values[start:end])
        y.append(abs(values[end]))
    return np.array(X, dtype=np.float64), np.array(y, dtype=np.float64)


X, y = create_windows(returns, WINDOW_SIZE)

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]

print(f"Train: {len(X_train)}  Val: {len(X_val)}")

Train: 2898  Val: 621


## Model 1: linear regression on all 30 raw values

`np.linalg.lstsq` solves for the best-fit coefficients directly (no training loop, no epochs, no dials to nudge - one calculation gives the exact best answer for a linear model). A column of 1s is added so the model can also fit an intercept (a baseline offset).

In [2]:
X_train_design = np.hstack([np.ones((len(X_train), 1)), X_train])
X_val_design = np.hstack([np.ones((len(X_val), 1)), X_val])

coeffs_full, *_ = np.linalg.lstsq(X_train_design, y_train, rcond=None)
predictions_full = X_val_design @ coeffs_full

rmse_full = np.sqrt(np.mean((predictions_full - y_val) ** 2))
print(f"Linear regression (30 raw values) RMSE: {rmse_full:.4f}")
print(f"Prediction std: {predictions_full.std():.4f}  (actual std: {y_val.std():.4f})")

Linear regression (30 raw values) RMSE: 0.5981
Prediction std: 0.0848  (actual std: 0.5875)


## Model 2: linear regression on one feature (recent realized volatility)

Simplify further: just the window's average absolute return as a single predictor - the most basic version of "recent volatility predicts near-future volatility," the same core idea behind GARCH's persistence (beta).

In [3]:
recent_vol_train = np.abs(X_train).mean(axis=1)
recent_vol_val = np.abs(X_val).mean(axis=1)

X_train_simple = np.column_stack([np.ones(len(X_train)), recent_vol_train])
X_val_simple = np.column_stack([np.ones(len(X_val)), recent_vol_val])

coeffs_simple, *_ = np.linalg.lstsq(X_train_simple, y_train, rcond=None)
predictions_simple = X_val_simple @ coeffs_simple

rmse_simple = np.sqrt(np.mean((predictions_simple - y_val) ** 2))
print(f"Linear regression (1 feature: recent realized vol) RMSE: {rmse_simple:.4f}")
print(f"Prediction std: {predictions_simple.std():.4f}  (actual std: {y_val.std():.4f})")
print(f"Coefficients (intercept, slope): {coeffs_simple}")

Linear regression (1 feature: recent realized vol) RMSE: 0.5868
Prediction std: 0.0975  (actual std: 0.5875)
Coefficients (intercept, slope): [0.23263922 0.67248756]


## The verdict

If the linear models also can't beat naive by much, that points to a genuine noise ceiling in this data/target - not a bug. If they clearly beat both naive and the transformer, that points to a fixable transformer-specific training problem.

In [4]:
naive_prediction = np.full_like(y_val, y_train.mean())
rmse_naive = np.sqrt(np.mean((naive_prediction - y_val) ** 2))

results = pd.DataFrame({
    "Model": ["Naive (train mean)", "Linear (30 raw values)", "Linear (1 feature: recent vol)", "Transformer (from notebook 11/12)"],
    "RMSE": [rmse_naive, rmse_full, rmse_simple, 0.5852],
})
results["Improvement over naive"] = (1 - results["RMSE"] / rmse_naive) * 100
results

,Model,RMSE,Improvement over naive
0,Naive (train mean),0.589738,0.000000
1,Linear (30 raw values),0.598058,-1.410696
2,Linear (1 feature: recent vol),0.586758,0.505356
3,Transformer (from notebook 11/12),0.585200,0.769526
